Limpieza.....

## 1. Carga de librerias

In [20]:
import pandas as pd
import numpy as np

## 2. Carga del dataset

In [21]:
df_sup_raw = pd.read_excel("../data_raw/superficie.xlsx", header=0)

## 3. Exploración inicial 

In [22]:
df_sup_raw.head()

,IGN_INFOGEO_MUNICIPIOS_filtrado_personalizado,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Identificador,Nombre,Población,Superficie (km2),Capital,Provincia,Ver en mapa
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,32003,A Arnoia,964,2069,Outeiro Cruz,Ourense,NaN
3,44001,Ababuj,75,543,Ababuj,Teruel,NaN
4,40001,Abades,858,3198,Abades,Segovia,NaN


In [23]:
df_sup_raw.columns

Index(['IGN_INFOGEO_MUNICIPIOS_filtrado_personalizado', 'Unnamed: 1',
       'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6'],
      dtype='object')

## 4. Arreglamos encabezado

In [24]:
# Usamos fila 0 como nombres de columnas
df_sup_raw.columns = df_sup_raw.iloc[0]

In [25]:
# Eliminamos las filas de encabezado
df_sup = df_sup_raw.iloc[2:].copy()

In [26]:
# Comprobamos
df_sup.head(), df_sup.columns, df_sup.shape

(0 Identificador    Nombre Población Superficie (km2)              Capital  \
 2         32003  A Arnoia       964             2069         Outeiro Cruz   
 3         44001    Ababuj        75              543               Ababuj   
 4         40001    Abades       858             3198               Abades   
 5         10001    Abadía       332             4508               Abadía   
 6         27001    Abadín     2 217            19556  Abadín o Provecende   
 
 0 Provincia Ver en mapa  
 2   Ourense         NaN  
 3    Teruel         NaN  
 4   Segovia         NaN  
 5   Cáceres         NaN  
 6      Lugo         NaN  ,
 Index(['Identificador', 'Nombre', 'Población', 'Superficie (km2)', 'Capital',
        'Provincia', 'Ver en mapa'],
       dtype='object', name=0),
 (8132, 7))

## 5. Seleccion de columnas necesarias

In [27]:
df_sup = df_sup[['Identificador', 'Nombre', 'Superficie (km2)']]

## 6. Renombrar columnas

In [28]:
# Para mantener constancia con los otros datasets
df_sup = df_sup.rename(columns={
    'Identificador': 'codigo_municipio',
    'Nombre': 'municipio',
    'Superficie (km2)': 'superficie_km2'
})

In [29]:
df_sup.columns

Index(['codigo_municipio', 'municipio', 'superficie_km2'], dtype='object', name=0)

## 7. Limpieza correcta de superficie

### El formato del dataset original utiliza:
- espacios como separador de miles
- coma como separador decimal

Python no reconoce este formato al convertir directamente a número, por lo que esos valores se transformaban en NaN.

Para solucionarlo, hemos limpiado el formato de los datos que aparecían de esa forma.

In [30]:
# Limpiamos formato de la superficie
df_sup['superficie_km2'] = (
    df_sup['superficie_km2']
    .astype(str)
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
)

## 8. Conversión de superficie a numérico 

In [31]:
df_sup['superficie_km2'] = pd.to_numeric(df_sup['superficie_km2'], errors="coerce")

In [32]:
df_sup['superficie_km2'].isna().sum()

np.int64(0)

## 9. Arreglamos escala de superficie

In [33]:
df_sup['superficie_km2'] = df_sup['superficie_km2'] / 100

## 10. Comprobación de duplicados

In [34]:
# Comprobamos duplicados, ya que solo debe haber una sola fila por municipio
df_sup.duplicated(subset=['codigo_municipio']).sum()


np.int64(0)

## 11. Ordenar el dataset

In [35]:
# Convertimos antes "codigo_municipio" a string
df_sup['codigo_municipio'] = df_sup['codigo_municipio'].astype(str).str.zfill(5)

In [36]:
df_sup = df_sup.sort_values(['codigo_municipio'])

In [37]:
df_sup.head()

,codigo_municipio,municipio,superficie_km2
348,01001,Alegría-Dulantzi,19.95
555,01002,Amurrio,96.19
621,01003,Aramaio,73.09
778,01004,Artziniega,27.29
726,01006,Armiñón,12.97


## 12. Exportación del dataset limpio

In [38]:
df_sup.to_csv("../data_processed/superficie_limpio.csv", index=False)